### Import Library & Inisialisasi Spark

In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
import pandas as pd
from google_play_scraper import Sort, reviews
from youtube_comment_downloader import YoutubeCommentDownloader
import time

# Memulai Spark Session
spark = SparkSession.builder \
    .appName("DataAcquisition") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark Session Berhasil Dimulai!")

Spark Session Berhasil Dimulai!


### Scraping Google Play Store (Multi-App)

In [ ]:
def scrape_playstore():
    apps = {
        'DANA': 'id.dana',
        'GoPay': 'com.gojek.app',
        'OVO': 'com.grb.ovo',
        'ShopeePay': 'com.shopee.id'
    }
    
    all_reviews = []
    target_per_app = 5000

    for name, app_id in apps.items():
        print(f"Menyedot SEMUA metadata ulasan untuk {name}...")
        result, _ = reviews(
            app_id, lang='id', country='id',
            sort=Sort.NEWEST, count=target_per_app
        )
        for r in result:
            r['source'] = 'PlayStore'
            r['app_name'] = name
            all_reviews.append(r)
            
    return pd.DataFrame(all_reviews)

df_ps = scrape_playstore()
print(f"Total Kolom Mentah PlayStore: {len(df_ps.columns)}")
df_ps.head(2)

Menyedot SEMUA metadata ulasan untuk DANA...
Menyedot SEMUA metadata ulasan untuk GoPay...
Menyedot SEMUA metadata ulasan untuk OVO...
Menyedot SEMUA metadata ulasan untuk ShopeePay...
Total Kolom Mentah PlayStore: 13


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,source,app_name
0,7086f77d-28de-41b2-ac88-42de80685d27,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,mantap,5,0,None,2026-05-12 14:06:26,None,NaT,None,PlayStore,DANA
1,8943d96a-83f9-43d7-a6c4-2f4bfbba021d,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,pokoknya mantap,5,0,2.123.2,2026-05-12 14:03:38,None,NaT,2.123.2,PlayStore,DANA


### Scraping YouTube Comments

In [3]:
def scrape_youtube():
    downloader = YoutubeCommentDownloader()
    urls = [
        "https://www.youtube.com/watch?v=sPbHjVND7fM",
        "https://www.youtube.com/watch?v=3nizazXJGMg",
        "https://www.youtube.com/watch?v=Xxnrd52GY6k"
    ]
    
    all_comments = []
    for url in urls:
        print(f"Menyedot SEMUA metadata komentar YouTube dari: {url}")
        comments = downloader.get_comments_from_url(url, sort_by=0)
        count = 0
        for comment in comments:
            # comment adalah raw dictionary dari YouTube API.
            comment['source'] = 'YouTube'
            comment['app_name'] = 'General'
            all_comments.append(comment)
            
            count += 1
            if count >= 500: break # Batas per video
            
    return pd.DataFrame(all_comments)

df_yt = scrape_youtube()
print(f"Total Kolom Mentah YouTube: {len(df_yt.columns)}")
df_yt.head(2)

Menyedot SEMUA metadata komentar YouTube dari: https://www.youtube.com/watch?v=sPbHjVND7fM
Menyedot SEMUA metadata komentar YouTube dari: https://www.youtube.com/watch?v=3nizazXJGMg
Menyedot SEMUA metadata komentar YouTube dari: https://www.youtube.com/watch?v=Xxnrd52GY6k
Total Kolom Mentah YouTube: 13


,cid,text,time,author,channel,votes,replies,photo,heart,reply,time_parsed,source,app_name
0,UgyBxIPMZYlkRrSy6894AaABAg,Seebank adminnya murah..kalau ewallet dana soa...,4 bulan yang lalu,@YadiLesmana-q4q,UC_tk9rq8QTDUn0Ca8_XqGUA,0,,https://yt3.ggpht.com/ficg-Rc_you7z0ww_rUXi9rG...,False,False,1.768285e+09,YouTube,General
1,UgzugxDR6duuSbJLUI14AaABAg,Ovo kalau kirim uang ke ovo itu dapat potongan...,3 minggu yang lalu,@ullaafrinabqary84,UC44QHSuQQASxwxQB5PL4b7A,0,1,https://yt3.ggpht.com/G7glFloA3daA2dY9CtXujqrm...,True,False,1.776838e+09,YouTube,General


### Penggabungan & Penyimpanan Data Mentah

In [4]:
# Menggabungkan data
df_final = pd.concat([df_ps, df_yt], ignore_index=True)

# Simpan ke CSV mentah
df_final.to_csv("raw_dataset_ewallet.csv", index=False)

print(f"Total Data Terkumpul: {len(df_final)} baris")

Total Data Terkumpul: 15014 baris


### Data Understanding

In [5]:
from pyspark.sql.functions import col, sum

# Membaca kembali menggunakan Spark
spark_df = spark.read.csv("raw_dataset_ewallet.csv", header=True, multiLine=True, escape='"')

print("Struktur Data (Schema):")
spark_df.printSchema()

print("Statistik Deskriptif Score (PlayStore Only):")
# Kita filter dan ubah tipe data score menjadi float agar bisa dihitung rata-ratanya
spark_df.filter(col("source") == "PlayStore") \
        .select(col("score").cast("float")) \
        .describe().show()

print("Pengecekan Missing Values per Kolom:")
spark_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in spark_df.columns]).show()

Struktur Data (Schema):
root
 |-- reviewId: string (nullable = true)
 |-- userName: string (nullable = true)
 |-- userImage: string (nullable = true)
 |-- content: string (nullable = true)
 |-- score: string (nullable = true)
 |-- thumbsUpCount: string (nullable = true)
 |-- reviewCreatedVersion: string (nullable = true)
 |-- at: string (nullable = true)
 |-- replyContent: string (nullable = true)
 |-- repliedAt: string (nullable = true)
 |-- appVersion: string (nullable = true)
 |-- source: string (nullable = true)
 |-- app_name: string (nullable = true)
 |-- cid: string (nullable = true)
 |-- text: string (nullable = true)
 |-- time: string (nullable = true)
 |-- author: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- votes: string (nullable = true)
 |-- replies: string (nullable = true)
 |-- photo: string (nullable = true)
 |-- heart: string (nullable = true)
 |-- reply: string (nullable = true)
 |-- time_parsed: string (nullable = true)

Statistik Deskriptif Sc